# Asistente virtual para tienda de videojuegos (Navi)
## Avance 1
**Autor:** Sebastian Martinez Maldonado  
**Matrícula:** 7302634

Este notebook contiene la estructura modular para el desarrollo de un chatbot basado en reglas, con pruebas de salida en cada celda para facilitar la documentación y captura de pantalla.


## 0. Instalación e importaciones

In [3]:
# Instalación de la librería principal de NLTK

!pip install nltk -q
!python --version

import re
import random
import unicodedata

# Clases específicas para dividir texto (tokens) y extraer raíces de palabras (stems)
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

import pandas as pd

print("Bibliotecas listas")

Python 3.13.16
Bibliotecas listas


## 1. Normalización del texto
Pasa todo a minúsculas, quita acentos y signos, y deja solo letras, números y espacios.

In [4]:
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-zñ0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

# Prueba de módulo 1
print(normalizar("¡¡HÓLÁ!! ¿Tienen juegos de Pokémon?"))

hola tienen juegos de pokemon


## 2. Tokenización y raíces (stemming)
Se separa el texto en palabras y cada palabra se reduce a su raíz (`precios` → `preci`).
La lista de palabras vacías es corta a propósito: la de NLTK incluye formas de "tener" (`tienen`, `tendrán`), que aquí son justamente la señal de la intención de existencias.

In [5]:
tokenizer = RegexpTokenizer(r"\w+")
stemmer = SnowballStemmer("spanish")

# Palabras sin contenido que solo meten ruido en la comparación
PALABRAS_VACIAS = {
    "a", "al", "algo", "ante", "como", "con", "contra", "cuando", "de", "del",
    "desde", "donde", "dos", "el", "ella", "ellas", "ellos", "en", "entre", "es",
    "esa", "ese", "eso", "esta", "estas", "este", "esto", "estos", "fue", "ha",
    "hasta", "la", "las", "le", "les", "lo", "los", "mas", "me", "mi", "mis",
    "mucho", "muy", "nada", "ni", "no", "nos", "o", "otra", "otro", "para",
    "pero", "poco", "por", "que", "se", "sin", "sobre", "su", "sus", "te",
    "todo", "todos", "tu", "tus", "un", "una", "uno", "unos", "unas", "y", "ya", "yo",
}

def extraer_raices(texto):
    tokens = tokenizer.tokenize(normalizar(texto))
    utiles = [t for t in tokens if t not in PALABRAS_VACIAS]
    return set(stemmer.stem(t) for t in utiles)

# Prueba  de módulo 2
print(extraer_raices("Buscando precios de videojuegos y consolas"))

{'consol', 'videojueg', 'busc', 'preci'}


## 3. Base de conocimiento
Dos piezas: el catálogo de productos (precio, existencias y palabras clave) y las intenciones con sus frases de ejemplo.
Las intenciones de precio, existencias y soporte no tienen respuestas fijas porque se arman con los datos del catálogo.

In [6]:
# Catálogo: es la información real que el bot puede consultar
catalogo = {
    "Nintendo Switch": {"precio": 6999, "stock": True, "palabras": [
        "switch", "nintendo", "nintendo switch", "nswitch",
        "suich", "swich", "switsh", "suitch",
        "consola de nintendo",
    ]},
    "PlayStation 5": {"precio": 11999, "stock": False, "palabras": [
        "ps5", "ps 5", "playstation", "playstation 5", "playstation5",
        "play station", "play 5", "ps cinco", "plei",
    ]},
    "Xbox One": {"precio": 5999, "stock": True, "palabras": [
        "xbox", "xbox one", "xboxone", "xbox 1", "xone",
        "x box", "x-box", "equis box",
    ]},
    "HyperX Alloy": {"precio": 1299, "stock": True, "palabras": [
        "teclado", "teclado gamer", "teclado mecanico", "mecanico",
        "keyboard", "hyperx", "hyper x", "hiperx", "hiper x",
        "alloy", "teclao",
    ]},
    "DualSense": {"precio": 1599, "stock": True, "palabras": [
        "dualsense", "dual sense", "dualsence", "dual sence",
        "control", "mando", "joystick", "gamepad",
        "control de ps5", "control de playstation", "mando de ps5",
        "control inalambrico", "dualshock",
    ]},
}


base_conocimiento = {
    "intenciones": {
        "saludo": {
            "ejemplos": ["hola", "buenos dias", "buenas tardes", "buenas noches", "que tal",
                         "necesito ayuda", "que puedes hacer", "que onda", "hey",
                         "buen dia", "buenas", "hola buenas", "holi", "ola", "que hubo",
                         "saludos", "como estas", "hay alguien", "alguien ahi",
                         "me puedes ayudar", "en que me ayudas", "con que me ayudas",
                         "como me puedes ayudar", "que sabes hacer", "que haces",
                         "quien eres", "que eres", "menu", "inicio", "empezar"],
            "respuestas": [
                "¡Hola! Soy Navi, tu asistente gamer. Puedo ayudarte con precios, existencias, horario o soporte.",
                "¡Qué gusto saludarte! Dime qué producto te interesa o qué necesitas revisar.",
                "¡Hey! Aquí Navi. Pregúntame por precios, existencias, horario o soporte de consolas.",
            ],
        },
        "consultar_horario": {
            "ejemplos": ["a que hora abren", "horario de atencion", "cierran tarde",
                         "desde que hora estan", "a que hora cierran", "hasta que hora abren", "hasta que hora estan",
                         "cuando abren", "cuando cierran", "horarios", "cual es su horario",
                         "que horario manejan", "estan abiertos", "estan abiertos ahorita",
                         "abren hoy", "abren los domingos", "abren el sabado",
                         "trabajan los domingos", "atienden los sabados", "dias de atencion",
                         "hora de apertura", "hora de cierre", "a que hora salen",
                         "abren manana", "abren temprano"],
            "respuestas": [
                "Nuestro horario es de lunes a sábado de 10:00 AM a 8:00 PM.",
                "Abrimos de lunes a sábado, de 10:00 AM a 8:00 PM.",
                "Te atendemos de lunes a sábado, de 10:00 AM a 8:00 PM. Los domingos permanecemos cerrados.",
            ],
        },
        "despedida": {
            "ejemplos": ["gracias", "adios", "nos vemos", "bye", "salir",
                         "muchas gracias", "ok gracias", "listo gracias", "grax",
                         "hasta luego", "hasta pronto", "hasta la proxima", "chao", "chau",
                         "me voy", "ya me voy", "eso es todo", "es todo", "nada mas",
                         "ya no necesito nada", "cuidate", "que descanses", "terminar"],
            "respuestas": [
                "¡Nos vemos! Que tengas una excelente partida.",
                "Gracias por escribir. ¡Hasta la próxima!",
                "¡Con gusto! Aquí estaré cuando necesites algo más.",
            ],
        },
        "consultar_precio": {
            "ejemplos": ["cuanto cuesta", "cual es el precio", "que valor tiene",
                         "como cuanto cuesta", "precio de",
                         "cuanto vale", "cuanto es", "cuanto sale", "cuanto me sale",
                         "cuanto cuestan", "que precio tiene", "que precio manejan",
                         "a como esta", "a como lo tienen", "en cuanto esta",
                         "cual es el costo", "costo de", "valor de", "precio",
                         "me dices el precio", "me pasas el precio", "cuanto cobran por",
                         "cuanto piden por", "de cuanto es", "cuanto estan pidiendo"],
        },
        "consultar_stock": {
            "ejemplos": ["tienen disponible", "hay stock", "busco el juego",
                         "de casualidad tendran", "tienen en existencia", "saben si hay",
                         "hay disponible", "esta disponible", "hay existencias",
                         "tienen en stock", "lo tienen", "lo tienen en tienda",
                         "queda alguno", "quedan", "tienen alguno", "tienen piezas",
                         "ya llego", "ya hay", "esta agotado", "se agoto",
                         "lo manejan", "lo venden", "estoy buscando", "tendran",
                         "tienen todavia", "hay en tienda"],
        },
        "soporte_tecnico": {
            "ejemplos": ["mi consola no prende", "se descompuso", "mantenimiento",
                         "dejo de servir", "murio mi consola", "hace ruidos extraños",
                         "no enciende", "no prende", "no funciona", "no sirve",
                         "esta fallando", "falla", "se apaga sola", "se apaga solo",
                         "se reinicia sola", "se congela", "se traba", "no carga",
                         "no lee los discos", "se sobrecalienta", "calienta mucho",
                         "no da imagen", "pantalla negra", "no conecta al wifi",
                         "el control no responde", "esta roto", "se daño",
                         "reparacion", "arreglar mi consola", "reparan consolas",
                         "servicio tecnico", "hace ruido raro", "huele a quemado"],
        },
    }
}

# Respuesta de seguridad cuando no se reconoce la intención
respuestas_fallback = [
    "No estoy seguro de haber entendido. Puedo ayudarte con precios, existencias, horario o soporte de consolas.",
    "No entendí esa consulta. Prueba preguntando por precio, existencias, horario o soporte.",
    "Mmm, eso no lo tengo claro. Pregúntame por precios, existencias, horario o soporte técnico.",
]

# Mensaje con el que arranca la conversación
MENSAJE_APERTURA = f"""
=============================================================
🤖 NAVI ONLINE - Asistente Virtual de Videojuegos
=============================================================
¡Hola! Soy Navi, tu asistente gamer. Estoy aquí para hacerte la vida más fácil.

Puedo ayudarte con lo siguiente:
 🎮 Consultar el PRECIO de consolas, accesorios y juegos.
 📦 Revisar el STOCK o la disponibilidad de un producto.
 🛠️ Orientarte sobre SOPORTE TÉCNICO y reparaciones.
 🕒 Informarte sobre nuestro HORARIO de atención.

Algunos de los productos que manejo: {", ".join(catalogo)}.
=============================================================
"""

print("Intenciones:", list(base_conocimiento["intenciones"].keys()))
print("Productos en catálogo:", list(catalogo.keys()))

Intenciones: ['saludo', 'consultar_horario', 'despedida', 'consultar_precio', 'consultar_stock', 'soporte_tecnico']
Productos en catálogo: ['Nintendo Switch', 'PlayStation 5', 'Xbox One', 'HyperX Alloy', 'DualSense']


## 4. Vocabulario de intenciones
Se juntan las raíces de todas las frases de ejemplo de cada intención.

In [7]:
vocabulario_intenciones = {}

for intencion, datos in base_conocimiento["intenciones"].items():
    raices = set()
    for ejemplo in datos["ejemplos"]:
        raices.update(extraer_raices(ejemplo))
    vocabulario_intenciones[intencion] = raices

# Prueba
print("consultar_precio:", sorted(vocabulario_intenciones["consultar_precio"]))
print("consultar_stock :", sorted(vocabulario_intenciones["consultar_stock"]))

consultar_precio: ['cobr', 'cost', 'cual', 'cuant', 'cuest', 'dic', 'estan', 'manej', 'pas', 'pid', 'preci', 'sal', 'tien', 'val', 'valor']
consultar_stock : ['agot', 'algun', 'busc', 'casual', 'dispon', 'estoy', 'existent', 'hay', 'jueg', 'lleg', 'manej', 'piez', 'qued', 'sab', 'si', 'stock', 'tendr', 'tien', 'tiend', 'todavi', 'vend']


## 5. Detección de intención
Se cuentan las raíces que comparte el mensaje con cada intención. Las palabras del producto no cuentan y el umbral se ajusta al tamaño del mensaje: uno de una sola palabra se acepta con una coincidencia; uno más largo necesita dos.

In [8]:
UMBRAL = 2

# Raíces de los productos: no cuentan como señal de la intención
RAICES_PRODUCTO = set()
for datos in catalogo.values():
    for palabra in datos["palabras"]:
        RAICES_PRODUCTO |= extraer_raices(palabra)

def detectar_intencion(mensaje):
    raices = extraer_raices(mensaje) - RAICES_PRODUCTO

    if not raices:
        return "desconocida", 0

    puntajes = {}
    for intencion, raices_intencion in vocabulario_intenciones.items():
        puntajes[intencion] = len(raices & raices_intencion)

    mejor_intencion = max(puntajes, key=puntajes.get)
    mejor_puntaje = puntajes[mejor_intencion]

    # Un mensaje de una sola palabra no puede alcanzar dos coincidencias
    umbral = min(UMBRAL, len(raices))

    if mejor_puntaje < umbral:
        return "desconocida", mejor_puntaje

    return mejor_intencion, mejor_puntaje

# Pruebas
for mensaje in ["¿Tienen en stock consolas Nintendo?", "¿A qué hora abren?", "Quiero devolver una compra"]:
    print(mensaje, "->", detectar_intencion(mensaje))

¿Tienen en stock consolas Nintendo? -> ('consultar_stock', 2)
¿A qué hora abren? -> ('consultar_horario', 2)
Quiero devolver una compra -> ('desconocida', 0)


## 6. Detección de producto
Se comparan las palabras del mensaje contra las palabras clave de cada producto del catálogo.
Se comparan frases completas, no pedazos de palabra (así "play" no coincide con "playera"), y gana la frase más larga.


In [9]:
# Lista de (frase normalizada, producto) para comparar frases completas
PRODUCTOS_POR_CLAVE = [
    (normalizar(palabra), producto)
    for producto, datos in catalogo.items()
    for palabra in datos["palabras"]
]

def detectar_producto(mensaje):
    texto = " " + normalizar(mensaje) + " "

    mejor_producto = None
    mejor_largo = 0

    # Gana la frase más larga: "control de ps5" antes que "ps5"
    for clave, producto in PRODUCTOS_POR_CLAVE:
        if " " + clave + " " in texto and len(clave) > mejor_largo:
            mejor_producto = producto
            mejor_largo = len(clave)

    return mejor_producto

# Pruebas
for mensaje in ["busco una nintendo switch", "mi xbox one no prende", "cuanto cuesta un teclado"]:
    print(mensaje, "->", detectar_producto(mensaje))

busco una nintendo switch -> Nintendo Switch
mi xbox one no prende -> Xbox One
cuanto cuesta un teclado -> HyperX Alloy


## 7. Generación de respuesta
Con la intención, el puntaje y el producto se arma la respuesta. Precio, existencias y soporte responden con datos del catálogo; el resto usa las respuestas fijas. Si falta el producto, el bot pregunta de cuál.

In [10]:
def generar_respuesta(mensaje):
    intencion, puntaje = detectar_intencion(mensaje)
    producto = detectar_producto(mensaje)

    if intencion == "desconocida":
        if producto:
            respuesta = f"¿Qué necesitas saber de {producto}? Puedo revisar precio, existencias o soporte."
        else:
            respuesta = random.choice(respuestas_fallback)

    elif intencion == "consultar_precio":
        if producto:
            precio = catalogo[producto]["precio"]
            respuesta = f"El precio de {producto} es ${precio:,}."
        else:
            respuesta = "¿De cuál producto? Por ahora manejo: " + ", ".join(catalogo) + "."

    elif intencion == "consultar_stock":
        if producto is None:
            respuesta = "¿De cuál producto quieres revisar existencias? Manejo: " + ", ".join(catalogo) + "."
        elif catalogo[producto]["stock"]:
            precio = catalogo[producto]["precio"]
            respuesta = f"Sí tenemos {producto} en existencia (${precio:,})."
        else:
            respuesta = f"Por ahora no tenemos {producto} en existencia."

    elif intencion == "soporte_tecnico":
        if producto:
            respuesta = f"Tráenos tu {producto} a la tienda para un diagnóstico sin costo."
        else:
            respuesta = "Tráenos tu consola o accesorio a la tienda para un diagnóstico sin costo."

    else:
        respuesta = random.choice(base_conocimiento["intenciones"][intencion]["respuestas"])

    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "producto": producto or "no especificado",
        "respuesta": respuesta,
    }

# Pruebas
for mensaje in ["cuanto cuesta la nintendo switch", "tienen ps5", "cuanto cuesta arreglarla"]:
    r = generar_respuesta(mensaje)
    print("Usuario:", mensaje)
    print("Navi   :", r["respuesta"], "\n")

Usuario: cuanto cuesta la nintendo switch
Navi   : El precio de Nintendo Switch es $6,999. 

Usuario: tienen ps5
Navi   : El precio de PlayStation 5 es $11,999. 

Usuario: cuanto cuesta arreglarla
Navi   : ¿De cuál producto? Por ahora manejo: Nintendo Switch, PlayStation 5, Xbox One, HyperX Alloy, DualSense. 



## 8. Quince conversaciones de prueba
Doce consultas dentro del alcance (cubren las siete intenciones) y tres fuera de alcance para comprobar el fallback.
La conversación arranca con un mensaje de apertura y después corre el lote completo.


In [11]:
casos = [
    {"mensaje": "¡Hola, muy buenos días!", "esperada": "saludo"},
    {"mensaje": "¿Tienen en stock consolas Nintendo Switch?", "esperada": "consultar_stock"},
    {"mensaje": "¿Y cuál es el precio de un teclado para PC?", "esperada": "consultar_precio"},
    {"mensaje": "OK, ¿a qué hora abren mañana?", "esperada": "consultar_horario"},
    {"mensaje": "Mi Xbox One hace ruidos extraños, se descompuso", "esperada": "soporte_tecnico"},
    {"mensaje": "¿De casualidad tendrán en bodega una PS5?", "esperada": "consultar_stock"},
    {"mensaje": "¿Cómo cuánto cuesta arreglarla?", "esperada": "consultar_precio"},
    {"mensaje": "¿Qué onda, necesito ayuda?", "esperada": "saludo"},
    {"mensaje": "¿Cierran tarde hoy?", "esperada": "consultar_horario"},
    {"mensaje": "Perfecto, gracias, adiós", "esperada": "despedida"},
    {"mensaje": "Quiero devolver una compra", "esperada": "desconocida"},
    {"mensaje": "¿Venden gift cards?", "esperada": "desconocida"},
    {"mensaje": "¿Cuánto vale el control de PS5?", "esperada": "consultar_precio"},
    {"mensaje": "No prende mi Nintendo Switch", "esperada": "soporte_tecnico"},
    {"mensaje": "¿Aceptan facturas?", "esperada": "desconocida"},
]

# Chat simulado: apertura y después las quince conversaciones
print(MENSAJE_APERTURA)

for i, caso in enumerate(casos, 1):
    r = generar_respuesta(caso["mensaje"])
    print(f"{i:2d}. Tú:   {caso['mensaje']}")
    print(f"    Navi: {r['respuesta']}")
    print(f"    [intención: {r['intencion']} | producto: {r['producto']}]\n")



🤖 NAVI ONLINE - Asistente Virtual de Videojuegos
¡Hola! Soy Navi, tu asistente gamer. Estoy aquí para hacerte la vida más fácil.

Puedo ayudarte con lo siguiente:
 🎮 Consultar el PRECIO de consolas, accesorios y juegos.
 📦 Revisar el STOCK o la disponibilidad de un producto.
 🛠️ Orientarte sobre SOPORTE TÉCNICO y reparaciones.
 🕒 Informarte sobre nuestro HORARIO de atención.

Algunos de los productos que manejo: Nintendo Switch, PlayStation 5, Xbox One, HyperX Alloy, DualSense.

 1. Tú:   ¡Hola, muy buenos días!
    Navi: ¡Hey! Aquí Navi. Pregúntame por precios, existencias, horario o soporte de consolas.
    [intención: saludo | producto: no especificado]

 2. Tú:   ¿Tienen en stock consolas Nintendo Switch?
    Navi: Sí tenemos Nintendo Switch en existencia ($6,999).
    [intención: consultar_stock | producto: Nintendo Switch]

 3. Tú:   ¿Y cuál es el precio de un teclado para PC?
    Navi: El precio de HyperX Alloy es $1,299.
    [intención: consultar_precio | producto: HyperX Allo

## 9. Métricas
Se calculan sobre las quince conversaciones.
- **Precisión de intención:** proporción de casos con la intención esperada.
- **Tasa de fallback:** proporción de mensajes que terminaron como `desconocida`.


In [12]:
filas = []

for caso in casos:
    r = generar_respuesta(caso["mensaje"])
    filas.append({
        "Mensaje": caso["mensaje"],
        "Esperada": caso["esperada"],
        "Detectada": r["intencion"],
        "Producto": r["producto"],
        "Respuesta": r["respuesta"],
        "Correcta": r["intencion"] == caso["esperada"],
    })

df = pd.DataFrame(filas)
precision = df["Correcta"].mean()
tasa_fallback = (df["Detectada"] == "desconocida").mean()

print(f"Precisión de intención: {precision:.1%} ({df['Correcta'].sum()}/{len(df)})")
print(f"Tasa de fallback: {tasa_fallback:.1%}")
df

Precisión de intención: 100.0% (15/15)
Tasa de fallback: 20.0%


,Mensaje,Esperada,Detectada,Producto,Respuesta,Correcta
0,"¡Hola, muy buenos días!",saludo,saludo,no especificado,"¡Hey! Aquí Navi. Pregúntame por precios, exist...",True
1,¿Tienen en stock consolas Nintendo Switch?,consultar_stock,consultar_stock,Nintendo Switch,"Sí tenemos Nintendo Switch en existencia ($6,9...",True
2,¿Y cuál es el precio de un teclado para PC?,consultar_precio,consultar_precio,HyperX Alloy,"El precio de HyperX Alloy es $1,299.",True
3,"OK, ¿a qué hora abren mañana?",consultar_horario,consultar_horario,no especificado,"Te atendemos de lunes a sábado, de 10:00 AM a ...",True
4,"Mi Xbox One hace ruidos extraños, se descompuso",soporte_tecnico,soporte_tecnico,Xbox One,Tráenos tu Xbox One a la tienda para un diagnó...,True
5,¿De casualidad tendrán en bodega una PS5?,consultar_stock,consultar_stock,PlayStation 5,Por ahora no tenemos PlayStation 5 en existencia.,True
6,¿Cómo cuánto cuesta arreglarla?,consultar_precio,consultar_precio,no especificado,¿De cuál producto? Por ahora manejo: Nintendo ...,True
7,"¿Qué onda, necesito ayuda?",saludo,saludo,no especificado,"¡Hey! Aquí Navi. Pregúntame por precios, exist...",True
8,¿Cierran tarde hoy?,consultar_horario,consultar_horario,no especificado,"Abrimos de lunes a sábado, de 10:00 AM a 8:00 PM.",True
9,"Perfecto, gracias, adiós",despedida,despedida,no especificado,Gracias por escribir. ¡Hasta la próxima!,True


## 10. Análisis de errores y casos límite
Cuatro mensajes que muestran dónde se rompe el enfoque por reglas.

In [13]:
def analizar(mensaje, nota):
    r = generar_respuesta(mensaje)
    print("Usuario:", mensaje)
    print("Intención:", r["intencion"], "| puntaje:", r["puntaje"], "| producto:", r["producto"])
    print("Navi:", r["respuesta"])
    print("Análisis:", nota, "\n")

# 1. Errores de ortografía: las coincidencias no alcanzan el umbral
analizar("ola ke oras esta avierto", "El bot no tolera tal cantidad de errores de escritura; cae en fallback.")

# 2. Dos intenciones en un mismo mensaje
analizar("hola, quiero saber a que hora abren para llevar mi nintendo que se descompuso",
         "Gana la intención con más coincidencias (horario), no la que el usuario quiso decir.")

# 3. Dos productos en una consulta
analizar("cuanto cuesta el xbox y la playstation",
         "Responde solo por el producto cuya frase coincide más larga en el catálogo (PlayStation 5).")

# 4. Precio sin producto
analizar("cuanto cuesta arreglarla",
         "No es un error: el bot pide el producto antes de responder con un precio.")

Usuario: ola ke oras esta avierto
Intención: desconocida | puntaje: 1 | producto: no especificado
Navi: No entendí esa consulta. Prueba preguntando por precio, existencias, horario o soporte.
Análisis: El bot no tolera tal cantidad de errores de escritura; cae en fallback. 

Usuario: hola, quiero saber a que hora abren para llevar mi nintendo que se descompuso
Intención: consultar_horario | puntaje: 3 | producto: Nintendo Switch
Navi: Abrimos de lunes a sábado, de 10:00 AM a 8:00 PM.
Análisis: Gana la intención con más coincidencias (horario), no la que el usuario quiso decir. 

Usuario: cuanto cuesta el xbox y la playstation
Intención: consultar_precio | puntaje: 2 | producto: PlayStation 5
Navi: El precio de PlayStation 5 es $11,999.
Análisis: Responde solo por el producto cuya frase coincide más larga en el catálogo (PlayStation 5). 

Usuario: cuanto cuesta arreglarla
Intención: consultar_precio | puntaje: 2 | producto: no especificado
Navi: ¿De cuál producto? Por ahora manejo: Nint

## 11. Chatbot - Conversación en vivo
La siguiente celda abre una conversación en tiempo real. Ejecútala por separado, no con «Ejecutar todo», porque el programa espera tu respuesta.


In [14]:
# Bucle interactivo para probar el bot en tiempo real
print("--- CHAT EN VIVO ---")
print("Para terminar la sesión, escribe \"Salir\" ")
print(MENSAJE_APERTURA)

while True:
    msg = input("👤 Tú: ")

    # Salir aunque escriban "Salir", "SALIR" o con espacios
    if normalizar(msg) == "salir":
        print("🤖 Navi: ¡Partida guardada! Hasta pronto.")
        break

    # Si no escribieron nada, no respondemos
    if not msg.strip():
        continue

    # generar_respuesta devuelve un diccionario: usamos solo la respuesta
    r = generar_respuesta(msg)
    print(f"🤖 Navi: {r['respuesta']}\n")


--- CHAT EN VIVO ---
Para terminar la sesión, escribe "Salir" 

🤖 NAVI ONLINE - Asistente Virtual de Videojuegos
¡Hola! Soy Navi, tu asistente gamer. Estoy aquí para hacerte la vida más fácil.

Puedo ayudarte con lo siguiente:
 🎮 Consultar el PRECIO de consolas, accesorios y juegos.
 📦 Revisar el STOCK o la disponibilidad de un producto.
 🛠️ Orientarte sobre SOPORTE TÉCNICO y reparaciones.
 🕒 Informarte sobre nuestro HORARIO de atención.

Algunos de los productos que manejo: Nintendo Switch, PlayStation 5, Xbox One, HyperX Alloy, DualSense.

👤 Tú: salir
🤖 Navi: ¡Partida guardada! Hasta pronto.
